In [0]:
data = [
    (1,"John",25000),
    (2,"Millinia",26000),
    (3,"Ravi",27000),
    (4,"Priya",28000)
]

sample_df = spark.createDataFrame(data,['id','name','salary'])

In [0]:
sample_df.printSchema()

In [0]:
spark.sql("use CATALOG `databricks-pyspark` ")

In [0]:
spark.sql("use SCHEMA spark_ns ")

In [0]:
sample_df.write.mode("overwrite").format('delta').saveAsTable("demo_schema_enforcement")


In [0]:
data = [
    (11,"vincent",45544,"USA"),
    (12,"Ravi",45544,"USA"),
    (13,"Priya",45544,"USA"),
  
]
extra_df = spark.createDataFrame(data,["id","name","salary","country"])

In [0]:
extra_df.write.mode("append").format("delta").saveAsTable("demo_schema_enforcement")

In [0]:
extra_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("demo_schema_enforcement")

In [0]:
spark.table("demo_schema_enforcement").show()
spark.table("demo_schema_enforcement").printSchema()

In [0]:
spark.sql("DROP TABLE IF EXISTS employee_target")
spark.sql("DROP TABLE IF EXISTS employee_source")

In [0]:
data = [
    (1,"John",25000),
    (2,"Rishi",32000)
]
source_data = spark.createDataFrame(data,['id','name','salary'])
source_data.write.mode("overwrite").format("delta").saveAsTable("employee_target")

In [0]:
data = [
    (2,"Linka",12000,"IT"),
    (3,"Revi",13000,"Sales")
]
source_df = spark.createDataFrame(data,['id','name','salary','dept'])

In [0]:
source_df.write.format("delta").mode("overwrite").saveAsTable("employee_source")

In [0]:
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

In [0]:
spark.sql("""
          MERGE WITH SCHEMA EVOLUTION INTO employee_target t 
          using employee_source s
          on s.id = t.id
          WHEN MATCHED THEN update SET *
          WHEN NOT MATCHED THEN insert *
          """)

In [0]:
spark.table("employee_target").show()